## Inspect Converted `data/set` Dataset

This notebook inspects the converted dataset stored in `../../data/set`.

Expected per-file payload:
- one `.npy` file per battery cell
- filename is a 4-character base62 content hash
- array shape is `(cycle, sample, channel)`
- channel mapping: `0 -> voltage_in_V`, `1 -> current_in_A`
- sampling is expected to be at `dt = 1s`

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

datapath = Path("../../data/set")
files = sorted(p for p in datapath.glob("*.npy"))

print(f"datapath: {datapath.resolve()}")
print(f"number of .npy files: {len(files)}")
print("first 10 files:", [p.name for p in files[:10]])

In [ ]:
assert files, "No .npy files found in ../../data/set"

example_path = files[0]
example = np.load(example_path, allow_pickle=False)

print("example file:", example_path.name)
print("dtype:", example.dtype)
print("shape:", example.shape)
assert example.ndim == 3, "Expected tensor shape (cycle, sample, channel)"
assert example.shape[-1] == 2, "Expected 2 channels: voltage/current"

print("example[0, :5, :]:")
print(example[0, :5, :])

### Dataset Consistency Summary

This section checks tensor dimensionality, cycle/sample sizes, and padding behavior (NaN tail padding).

In [ ]:
from statistics import median

cell_rows = []
all_cycle_lengths = []
all_tail_nan_fraction = []
bad_shapes = []

for path in files:
    arr = np.load(path, allow_pickle=False)

    if arr.ndim != 3 or arr.shape[-1] != 2:
        bad_shapes.append((path.name, arr.shape))
        continue

    cycles, max_samples, _ = arr.shape
    finite_mask = np.isfinite(arr[:, :, 0]) & np.isfinite(arr[:, :, 1])
    valid_lengths = finite_mask.sum(axis=1).astype(int)

    cycle_rows = []
    for valid_len in valid_lengths:
        all_cycle_lengths.append(int(valid_len))
        tail_n = max_samples - int(valid_len)
        all_tail_nan_fraction.append(tail_n / max_samples if max_samples else 0.0)
        cycle_rows.append(int(valid_len))

    cell_rows.append(
        {
            "file": path.name,
            "cycles": cycles,
            "max_samples": max_samples,
            "min_valid_samples": min(cycle_rows) if cycle_rows else 0,
            "median_valid_samples": int(np.median(cycle_rows)) if cycle_rows else 0,
            "max_valid_samples": max(cycle_rows) if cycle_rows else 0,
        }
    )

print(f"files inspected: {len(files)}")
print(f"files with valid shape: {len(cell_rows)}")
print(f"files with bad shape: {len(bad_shapes)}")
if bad_shapes:
    print("bad shape examples:", bad_shapes[:5])

if cell_rows:
    cycle_counts = [row["cycles"] for row in cell_rows]
    max_samples_per_cell = [row["max_samples"] for row in cell_rows]

    print(
        "cycles per cell stats:",
        f"min={min(cycle_counts)}, median={median(cycle_counts):.1f}, max={max(cycle_counts)}",
    )
    print(
        "max sample length per cell stats:",
        f"min={min(max_samples_per_cell)}, median={median(max_samples_per_cell):.1f}, max={max(max_samples_per_cell)}",
    )

if all_cycle_lengths:
    print(
        "valid samples per cycle stats:",
        f"min={min(all_cycle_lengths)}, median={median(all_cycle_lengths):.1f}, max={max(all_cycle_lengths)}",
    )

if all_tail_nan_fraction:
    print(
        "tail padding fraction stats:",
        f"min={min(all_tail_nan_fraction):.4f}, median={median(all_tail_nan_fraction):.4f}, max={max(all_tail_nan_fraction):.4f}",
    )

### Training-Oriented Capacity Planning

The next cells estimate:

1. a practical `max_signal_positions` value from dataset signal-length statistics and the configured Conv1D downsampling stack
2. the maximum available `epoch_samples` per split (`train`/`val`/`test`) using the same split seed and ratios as training

In [ ]:
import math
from pathlib import Path

import yaml

# Load training defaults so this notebook stays aligned with training config.
cfg_path = Path("../../configs/default.yaml")
with cfg_path.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

encoder_cfg = cfg["encoder"]
conv_kernels = encoder_cfg["conv_kernels"]
conv_strides = encoder_cfg["conv_strides"]


def conv1d_out_len(
    length: int,
    kernel: int,
    stride: int,
    padding: int,
    dilation: int = 1,
) -> int:
    return math.floor((length + 2 * padding - dilation * (kernel - 1) - 1) / stride + 1)


def downsampled_len(length: int) -> int:
    out = int(length)
    for kernel, stride in zip(conv_kernels, conv_strides, strict=True):
        out = conv1d_out_len(out, kernel=kernel, stride=stride, padding=kernel // 2)
    return max(out, 0)


if not all_cycle_lengths:
    raise RuntimeError(
        "Run the dataset consistency summary cell first to populate all_cycle_lengths."
    )

signal_lengths = np.asarray(all_cycle_lengths, dtype=np.int64)
down_lengths = np.asarray(
    [downsampled_len(int(v)) for v in signal_lengths], dtype=np.int64
)

q95 = int(np.percentile(down_lengths, 95))
q99 = int(np.percentile(down_lengths, 99))
q999 = int(np.percentile(down_lengths, 99.9))
max_len = int(down_lengths.max())

recommended = int(min(max_len, math.ceil(q999 * 1.1)))

print("config max_signal_positions:", encoder_cfg["max_signal_positions"])
print("downsampled lengths stats:")
print(
    {
        "min": int(down_lengths.min()),
        "median": int(np.median(down_lengths)),
        "p95": q95,
        "p99": q99,
        "p99_9": q999,
        "max": max_len,
    }
)
print("recommended max_signal_positions:", recommended)
if encoder_cfg["max_signal_positions"] < max_len:
    print(
        "WARNING: current max_signal_positions is lower than observed max downsampled length."
    )

In [ ]:
import random


def split_files_training_style(
    input_files: list[Path],
    seed: int,
    val_fraction: float,
    test_fraction: float,
) -> tuple[list[Path], list[Path], list[Path]]:
    ordered = sorted(input_files)
    rng = random.Random(seed)
    rng.shuffle(ordered)

    n_total = len(ordered)
    n_test = max(1, round(n_total * test_fraction)) if test_fraction > 0 else 0
    n_val = max(1, round(n_total * val_fraction)) if val_fraction > 0 else 0

    if n_val + n_test >= n_total:
        overflow = n_val + n_test - (n_total - 1)
        if overflow > 0:
            if n_val >= n_test and n_val > 1:
                n_val -= overflow
            elif n_test > 1:
                n_test -= overflow

    test_files = ordered[:n_test]
    val_files = ordered[n_test : n_test + n_val]
    train_files = ordered[n_test + n_val :]
    return train_files, val_files, test_files


def valid_cycle_count_for_file(path: Path) -> int:
    arr = np.load(path, allow_pickle=False)
    if arr.ndim != 3 or arr.shape[-1] != 2:
        return 0

    valid_cycles = 0
    min_q = float(cfg["data"]["min_discharge_capacity_ah"])
    dt_seconds = float(cfg["data"]["dt_seconds"])

    for cycle_idx in range(arr.shape[0]):
        cycle = arr[cycle_idx]
        valid_mask = np.isfinite(cycle[:, 0]) & np.isfinite(cycle[:, 1])
        if valid_mask.sum() <= 0:
            continue
        current = cycle[valid_mask, 1]
        discharge_capacity_ah = float(
            np.clip(-current, 0.0, None).sum() * dt_seconds / 3600.0
        )

        if cfg["data"]["drop_cycles_without_discharge"]:
            if discharge_capacity_ah >= min_q:
                valid_cycles += 1
        else:
            valid_cycles += 1

    return valid_cycles


def count_windows(
    file_group: list[Path], cycle_window: int, min_observed_cycles: int
) -> int:
    total = 0
    per_file = {}
    for path in file_group:
        n_cycles = valid_cycle_count_for_file(path)
        if n_cycles < min_observed_cycles:
            windows = 0
        elif n_cycles <= cycle_window:
            windows = 1
        else:
            windows = n_cycles - cycle_window + 1
        per_file[path.name] = windows
        total += windows
    return total, per_file


data_cfg = cfg["data"]
train_files, val_files, test_files = split_files_training_style(
    files,
    seed=int(data_cfg["split_seed"]),
    val_fraction=float(data_cfg["val_fraction"]),
    test_fraction=float(data_cfg["test_fraction"]),
)

cycle_window = int(data_cfg["cycle_window"])
min_observed_cycles = int(data_cfg["min_observed_cycles"])

train_max, train_windows_by_file = count_windows(
    train_files, cycle_window, min_observed_cycles
)
val_max, val_windows_by_file = count_windows(
    val_files, cycle_window, min_observed_cycles
)
test_max, test_windows_by_file = count_windows(
    test_files, cycle_window, min_observed_cycles
)

print("split battery counts:")
print({"train": len(train_files), "val": len(val_files), "test": len(test_files)})

print("max epoch_samples by split (full available windows):")
print({"train": train_max, "val": val_max, "test": test_max})

print("configured epoch_samples(train):", data_cfg["epoch_samples"])
if data_cfg["epoch_samples"] is not None and int(data_cfg["epoch_samples"]) > train_max:
    print(
        "NOTE: configured train epoch_samples is larger than available train windows; sampling will use replacement."
    )

In [ ]:
if cell_rows:
    fig_cycles = go.Figure(
        data=[go.Histogram(x=[r["cycles"] for r in cell_rows], nbinsx=60)]
    )
    fig_cycles.update_layout(
        title="Cycles Per Cell Distribution",
        xaxis_title="Cycles",
        yaxis_title="Count",
        template="plotly_white",
    )
    fig_cycles.show()

if all_cycle_lengths:
    fig_len = go.Figure(data=[go.Histogram(x=all_cycle_lengths, nbinsx=120)])
    fig_len.update_layout(
        title="Valid Samples Per Cycle Distribution",
        xaxis_title="Valid samples in cycle (dt=1s)",
        yaxis_title="Count",
        template="plotly_white",
    )
    fig_len.show()

### Cell-Level Signal Browser

Interactive browser for voltage/current trajectories by file and cycle index.
The x-axis is interpreted as time in seconds because sampling is uniform at `dt=1s`.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

tensor_cache = {}


def load_tensor(file_index: int) -> np.ndarray:
    if file_index not in tensor_cache:
        tensor_cache[file_index] = np.load(files[file_index], allow_pickle=False)
    return tensor_cache[file_index]


def get_cycle_valid_length(cycle_array: np.ndarray) -> int:
    finite = np.isfinite(cycle_array[:, 0]) & np.isfinite(cycle_array[:, 1])
    return int(finite.sum())


def make_cycle_figure(arr: np.ndarray, cycle_index: int, file_name: str):
    cycle = arr[cycle_index]
    n_valid = get_cycle_valid_length(cycle)
    if n_valid <= 0:
        n_valid = len(cycle)

    cycle_valid = cycle[:n_valid]
    t = np.arange(n_valid, dtype=np.int32)

    voltage = cycle_valid[:, 0]
    current = cycle_valid[:, 1]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=t, y=voltage, mode="lines", name="voltage_in_V", line={"color": "#377eb8"}
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t,
            y=current,
            mode="lines",
            name="current_in_A",
            line={"color": "#e41a1c"},
            yaxis="y2",
        )
    )

    fig.update_layout(
        title=f"{file_name} | cycle_index={cycle_index} | valid_samples={n_valid}",
        xaxis_title="time (s)",
        yaxis={"title": "voltage (V)"},
        yaxis2={"title": "current (A)", "overlaying": "y", "side": "right"},
        template="plotly_white",
        height=520,
    )
    return fig


file_slider = widgets.IntSlider(
    value=0, min=0, max=max(0, len(files) - 1), step=1, description="Cell"
)
cycle_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Cycle")
meta_html = widgets.HTML()
output = widgets.Output()


def clamp_cycle_slider(file_index: int):
    arr = load_tensor(file_index)
    n_cycles = arr.shape[0] if arr.ndim == 3 else 1
    cycle_slider.max = max(0, n_cycles - 1)
    cycle_slider.value = min(cycle_slider.value, cycle_slider.max)


def render(*_):
    file_index = file_slider.value
    arr = load_tensor(file_index)
    file_name = files[file_index].name

    if arr.ndim != 3 or arr.shape[-1] != 2:
        meta_html.value = f"<b>{file_name}</b> has invalid shape: {arr.shape}"
        with output:
            clear_output(wait=True)
            print("Invalid tensor shape for this file.")
        return

    clamp_cycle_slider(file_index)
    cidx = cycle_slider.value
    n_valid = get_cycle_valid_length(arr[cidx])
    meta_html.value = (
        f"<b>File:</b> {file_name}"
        f" &nbsp; <b>Shape:</b> {arr.shape}"
        f" &nbsp; <b>Cycle:</b> {cidx + 1}/{arr.shape[0]}"
        f" &nbsp; <b>Valid samples:</b> {n_valid}"
    )

    fig = make_cycle_figure(arr, cidx, file_name)
    with output:
        clear_output(wait=True)
        fig.show()


file_slider.observe(render, names="value")
cycle_slider.observe(render, names="value")
render()

display(widgets.VBox([file_slider, cycle_slider, meta_html, output]))

### Optional Integrity Spot-Check

Validate filename convention (`^[0-9A-Za-z]{4}\.npy$`) and check for exact duplicate tensors.

In [ ]:
import hashlib
import re

name_pattern = re.compile(r"^[0-9A-Za-z]{4}\.npy$")
bad_names = [p.name for p in files if not name_pattern.match(p.name)]
print("bad names:", bad_names[:10], "(total=", len(bad_names), ")")

seen = {}
duplicate_pairs = []
for path in files:
    arr = np.load(path, allow_pickle=False)
    key = hashlib.sha256(arr.tobytes(order="C")).hexdigest()
    if key in seen:
        duplicate_pairs.append((seen[key], path.name))
    else:
        seen[key] = path.name

print(f"exact duplicate tensors found: {len(duplicate_pairs)}")
if duplicate_pairs:
    print("examples:", duplicate_pairs[:10])